In [1]:
%pip install -q transformers accelerate rank-bm25 peft trl datasets pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 15.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import json
import glob
import time
import random
import string
from collections import Counter

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda"
    print("GPU:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    print("GPU unavailable. Training and inference may be slow.")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

GPU: Tesla T4
PyTorch: 2.11.0+cu128
CUDA available: True


In [3]:
DATA_ROOT = (
    "/kaggle/input/datasets/"
    "shubhamdineshjoshi/checkpoints-and-data"
)

PASSAGES_PATH = os.path.join(
    DATA_ROOT, "data/processed/passages.jsonl"
)

# We will set these after inspecting the available files.
TRAIN_PATH = None
VALIDATION_PATH = None
TEST_PATH = None

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

OUTPUT_DIR = "/kaggle/working/search_rl_project"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Dataset root exists:", os.path.exists(DATA_ROOT))
print("Passage file exists:", os.path.exists(PASSAGES_PATH))
print("Output directory:", OUTPUT_DIR)

Dataset root exists: True
Passage file exists: True
Output directory: /kaggle/working/search_rl_project


In [4]:
for root in ["/kaggle/input"]:
    print("\nINPUT FILES")
    for path in glob.glob(root + "/**/*", recursive=True):
        if os.path.isfile(path):
            name = os.path.basename(path).lower()

            if (
                name.endswith((".jsonl", ".json", ".parquet", ".csv"))
                or "train" in name
                or "valid" in name
                or "test" in name
                or "passage" in name
            ):
                print(path)


INPUT FILES
/kaggle/input/datasets/frankossai/natural-questions-dataset/Natural-Questions-Base.csv
/kaggle/input/datasets/frankossai/natural-questions-dataset/Natural-Questions-Filtered.csv
/kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/__huggingface_repos__.json
/kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/Training-Large-Language-Models-to-Reason-and-Leverage-Search-Engines-with-Reinforcement-Learning/Training-Large-Language-Models-to-Reason-and-Leverage-Search-Engines-with-Reinforcement-Learning/Training-Large-Language-Models-to-Reason-and-Leverage-Search-Engines-with-Reinforcement-Learning/scripts/prepare_passages.py
/kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/Training-Large-Language-Models-to-Reason-and-Leverage-Search-Engines-with-Reinforcement-Learning/Training-Large-Language-Models-to-Reason-and-Leverage-Search-Engines-with-Reinforcement-Learning/Training-Large-Language-Models-to-Reason-and-Leverage-Search-Engines-with-Reinfo

In [5]:
def read_jsonl(path, limit=None):
    records = []

    with open(path, "r", encoding="utf-8") as file:
        for i, line in enumerate(file):
            if limit is not None and i >= limit:
                break

            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def inspect_jsonl(path, n=2):
    print("File:", path)

    if not os.path.exists(path):
        print("FILE NOT FOUND")
        return

    records = read_jsonl(path, limit=n)

    print("Sample records:", len(records))

    for i, record in enumerate(records):
        print(f"\nRecord {i + 1}")
        print(json.dumps(record, indent=2, ensure_ascii=False)[:5000])


# Inspect the existing validation and passage files.
if os.path.exists(os.path.join(DATA_ROOT, "data/processed/validation.jsonl")):
    inspect_jsonl(
        os.path.join(DATA_ROOT, "data/processed/validation.jsonl")
    )

if os.path.exists(PASSAGES_PATH):
    inspect_jsonl(PASSAGES_PATH)

File: /kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/data/processed/validation.jsonl
Sample records: 2

Record 1
{
  "id": "6239",
  "question": "how many times has widespread panic played red rocks",
  "answers": [
    "54"
  ],
  "split": "validation"
}

Record 2
{
  "id": "50445",
  "question": "who played spencers mom in pretty little liars",
  "answers": [
    "Lesley Era"
  ],
  "split": "validation"
}
File: /kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/data/processed/passages.jsonl
Sample records: 2

Record 1
{
  "id": "nq-0-0",
  "text": "A common example of permission marketing is a newsletter sent to an advertising firm's customers. Such newsletters inform customers of upcoming events or promotions, or new products. In this type of advertising, a company that wants to send a newsletter to their customers may ask them at the point of purchase if they would like to receive the newsletter."
}

Record 2
{
  "id": "nq-1-0",
  "text": "Tracy McConne

In [6]:
all_jsonl = glob.glob(
    "/kaggle/input/**/*.jsonl",
    recursive=True
)

print("Potential split and corpus files:\n")

for path in sorted(all_jsonl):
    name = os.path.basename(path).lower()

    if any(word in name for word in [
        "train", "valid", "dev", "test", "passage", "natural"
    ]):
        print(path)

Potential split and corpus files:

/kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/Training-Large-Language-Models-to-Reason-and-Leverage-Search-Engines-with-Reinforcement-Learning/data/processed/passages.jsonl
/kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/Training-Large-Language-Models-to-Reason-and-Leverage-Search-Engines-with-Reinforcement-Learning/data/processed/train.jsonl
/kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/Training-Large-Language-Models-to-Reason-and-Leverage-Search-Engines-with-Reinforcement-Learning/data/processed/validation.jsonl
/kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/data/processed/passages.jsonl
/kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/data/processed/train.jsonl
/kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/data/processed/validation.jsonl


In [7]:
import os
import json

files_to_check = {
    "Processed train": os.path.join(DATA_ROOT, "data/processed/train.jsonl"),
    "Processed validation": os.path.join(DATA_ROOT, "data/processed/validation.jsonl"),
    "Wikipedia/passages corpus": PASSAGES_PATH,
}

def count_jsonl_lines(path):
    count = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                count += 1
    return count

for name, path in files_to_check.items():
    if os.path.exists(path):
        print(f"\n{name}")
        print("Path:", path)
        print("Records:", count_jsonl_lines(path))
        print("Size (MB):", round(os.path.getsize(path) / (1024**2), 2))
    else:
        print(f"\n{name}: NOT FOUND")


Processed train
Path: /kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/data/processed/train.jsonl
Records: 5000
Size (MB): 0.68

Processed validation
Path: /kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/data/processed/validation.jsonl
Records: 500
Size (MB): 0.07

Wikipedia/passages corpus
Path: /kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/data/processed/passages.jsonl
Records: 6453
Size (MB): 2.86


In [8]:
RAW_ROOT = "/kaggle/input/datasets/frankossai/natural-questions-dataset"

for filename in [
    "Natural-Questions-Base.csv",
    "Natural-Questions-Filtered.csv"
]:
    path = os.path.join(RAW_ROOT, filename)

    print("\n", filename)

    df_sample = pd.read_csv(path, nrows=5)

    print("Columns:", df_sample.columns.tolist())
    print("Sample:")
    display(df_sample.head(2))

    total_rows = sum(
        1 for _ in open(path, "r", encoding="utf-8")
    ) - 1

    print("Total rows:", total_rows)


 Natural-Questions-Base.csv
Columns: ['question', 'long_answers', 'short_answers']
Sample:


,question,long_answers,short_answers
0,which is the most common use of opt-in e-mail ...,<P> A common example of permission marketing i...,a newsletter sent to an advertising firm 's cu...
1,how i.met your mother who is the mother,"<P> Tracy McConnell , better known as `` The M...",Tracy McConnell


Total rows: 307373

 Natural-Questions-Filtered.csv
Columns: ['question', 'long_answers', 'short_answers']
Sample:


,question,long_answers,short_answers
0,which is the most common use of opt-in e-mail ...,A common example of permission marketing is a ...,A newsletter sent to an advertising firm's cus...
1,how i.met your mother who is the mother,"Tracy McConnell, better known as `` The Mother...",Tracy McConnell


Total rows: 86212


In [9]:
ARTIFACT_ROOT = os.path.join(DATA_ROOT, "artifacts")

for folder in [
    "qwen05b-search-rl",
    "qwen05b-search-rl-500-50",
    "qwen05b-search-rl-smoke-100-20"
]:
    path = os.path.join(ARTIFACT_ROOT, folder)

    print("\n" + "=" * 60)
    print("CHECKPOINT:", folder)

    manifest_path = os.path.join(path, "training_manifest.json")

    if os.path.exists(manifest_path):
        with open(manifest_path, "r") as f:
            print("Training manifest:")
            print(json.dumps(json.load(f), indent=2))

    metrics_path = os.path.join(path, "rl_metrics.json")

    if os.path.exists(metrics_path):
        with open(metrics_path, "r") as f:
            print("RL metrics:")
            print(json.dumps(json.load(f), indent=2))

    checkpoint_dirs = glob.glob(
        os.path.join(path, "checkpoint-*")
    )

    print("Checkpoint directories:", checkpoint_dirs)


CHECKPOINT: qwen05b-search-rl
Training manifest:
{
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "records": 100,
  "max_steps": 20,
  "num_generations": 2,
  "top_k": 3,
  "mode": "fixed-context GRPO smoke test"
}
Checkpoint directories: ['/kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/artifacts/qwen05b-search-rl/checkpoint-20']

CHECKPOINT: qwen05b-search-rl-500-50
Training manifest:
{
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "records": 500,
  "max_steps": 50,
  "num_generations": 2,
  "top_k": 3,
  "mode": "fixed-context GRPO smoke test"
}
Checkpoint directories: ['/kaggle/input/datasets/shubhamdineshjoshi/checkpoints-and-data/artifacts/qwen05b-search-rl-500-50/checkpoint-50']

CHECKPOINT: qwen05b-search-rl-smoke-100-20
Training manifest:
{
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "records": 100,
  "max_steps": 20,
  "num_generations": 2,
  "top_k": 3,
  "mode": "fixed-context GRPO smoke test"
}
Checkpoint directories: ['/kaggle/input/datasets/shubhamdineshjoshi/che

In [10]:
%pip install -q datasets rank-bm25 transformers accelerate

Note: you may need to restart the kernel to use updated packages.


In [11]:
from datasets import load_dataset
import itertools

WIKI_CONFIG = "20231101.en"

wiki_stream = load_dataset(
    "wikimedia/wikipedia",
    WIKI_CONFIG,
    split="train",
    streaming=True
)

# Inspect one article before processing
first_article = next(iter(wiki_stream))

print("Available fields:", first_article.keys())
print("\nTitle:", first_article.get("title"))
print("\nText sample:", first_article.get("text", "")[:1000])

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Available fields: dict_keys(['id', 'url', 'title', 'text'])

Title: Anarchism

Text sample: Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims maintain unnecessary coercion and hierarchy, typically including nation-states, and capitalism. Anarchism advocates for the replacement of the state with stateless societies and voluntary free associations. As a historically left-wing movement, this reading of anarchism is placed on the farthest left of the political spectrum, usually described as the libertarian wing of the socialist movement (libertarian socialism).

Humans have lived in societies without formal hierarchies long before the establishment of states, realms, or empires. With the rise of organised hierarchical bodies, scepticism toward authority also rose. Although traces of anarchist ideas are found all throughout history, modern anarchism emerged from the Enlightenment. During the

In [12]:
import os
import re
import json
from tqdm.auto import tqdm

WIKI_DIR = "/kaggle/working/search_rl_project/wiki"
os.makedirs(WIKI_DIR, exist_ok=True)

WIKI_PASSAGES_PATH = os.path.join(
    WIKI_DIR,
    "wikipedia_passages.jsonl"
)

MAX_ARTICLES = 50000
PASSAGE_WORDS = 120
OVERLAP_WORDS = 30

def clean_wiki_text(text):
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def split_into_passages(text, passage_words=120, overlap_words=30):
    words = text.split()

    if not words:
        return []

    passages = []
    step = passage_words - overlap_words

    for start in range(0, len(words), step):
        chunk = words[start:start + passage_words]

        if len(chunk) < 30:
            continue

        passages.append(" ".join(chunk))

        if start + passage_words >= len(words):
            break

    return passages

In [13]:
wiki_stream = load_dataset(
    "wikimedia/wikipedia",
    WIKI_CONFIG,
    split="train",
    streaming=True
)

article_count = 0
passage_count = 0

with open(WIKI_PASSAGES_PATH, "w", encoding="utf-8") as f:

    for article in tqdm(
        itertools.islice(wiki_stream, MAX_ARTICLES),
        total=MAX_ARTICLES
    ):

        title = article.get("title", "")
        text = clean_wiki_text(article.get("text", ""))

        if not title or not text:
            continue

        passages = split_into_passages(
            text,
            PASSAGE_WORDS,
            OVERLAP_WORDS
        )

        for index, passage in enumerate(passages):

            record = {
                "id": f"wiki-{article_count}-{index}",
                "title": title,
                "text": passage
            }

            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            passage_count += 1

        article_count += 1

print("Articles processed:", article_count)
print("Wikipedia passages created:", passage_count)
print("Output:", WIKI_PASSAGES_PATH)
print(
    "File size (MB):",
    round(os.path.getsize(WIKI_PASSAGES_PATH) / (1024**2), 2)
)

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

  0%|          | 0/50000 [00:00<?, ?it/s]

Articles processed: 50000
Wikipedia passages created: 469288
Output: /kaggle/working/search_rl_project/wiki/wikipedia_passages.jsonl
File size (MB): 354.56


In [14]:
wiki_records = read_jsonl(
    WIKI_PASSAGES_PATH,
    limit=3
)

for record in wiki_records:
    print("\nID:", record["id"])
    print("Title:", record["title"])
    print("Text:", record["text"][:500])


ID: wiki-0-0
Title: Anarchism
Text: Anarchism is a political philosophy and movement that is skeptical of all justifications for authority and seeks to abolish the institutions it claims maintain unnecessary coercion and hierarchy, typically including nation-states, and capitalism. Anarchism advocates for the replacement of the state with stateless societies and voluntary free associations. As a historically left-wing movement, this reading of anarchism is placed on the farthest left of the political spectrum, usually described as

ID: wiki-0-1
Title: Anarchism
Text: establishment of states, realms, or empires. With the rise of organised hierarchical bodies, scepticism toward authority also rose. Although traces of anarchist ideas are found all throughout history, modern anarchism emerged from the Enlightenment. During the latter half of the 19th and the first decades of the 20th century, the anarchist movement flourished in most parts of the world and had a significant role in worker

In [15]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import re
import os
import json

RAW_CSV = os.path.join(
    RAW_ROOT,
    "Natural-Questions-Base.csv"
)

nq_df = pd.read_csv(RAW_CSV)

print("Original rows:", len(nq_df))
print("Columns:", nq_df.columns.tolist())

def clean_answer(value):
    if pd.isna(value):
        return ""

    value = str(value)
    value = re.sub(r"<[^>]+>", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()

nq_df["question"] = nq_df["question"].fillna("").astype(str).str.strip()
nq_df["short_answer"] = nq_df["short_answers"].apply(clean_answer)
nq_df["long_answer"] = nq_df["long_answers"].apply(clean_answer)

nq_df = nq_df[nq_df["question"] != ""].copy()

print("\nTotal questions:", len(nq_df))
print("Questions with short answers:", (nq_df["short_answer"] != "").sum())
print("Questions without short answers:", (nq_df["short_answer"] == "").sum())

display(nq_df[["question", "short_answer", "long_answer"]].head())

Original rows: 307373
Columns: ['question', 'long_answers', 'short_answers']

Total questions: 307373
Questions with short answers: 106915
Questions without short answers: 200458


,question,short_answer,long_answer
0,which is the most common use of opt-in e-mail ...,a newsletter sent to an advertising firm 's cu...,A common example of permission marketing is a ...
1,how i.met your mother who is the mother,Tracy McConnell,"Tracy McConnell , better known as `` The Mothe..."
2,what type of fertilisation takes place in humans,,The process of fertilization involves a sperm ...
3,who had the most wins in the nfl,Tom Brady,Active quarterback Tom Brady holds the records...
4,what happened to the lost settlement of roanoke,,


In [16]:
qa_df = nq_df[nq_df["short_answer"] != ""].copy()

train_df, temp_df = train_test_split(
    qa_df,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))
print("Total used:", len(train_df) + len(val_df) + len(test_df))

Training: 96223
Validation: 5346
Testing: 5346
Total used: 106915


In [17]:
NQ_DIR = "/kaggle/working/search_rl_project/nq_splits"
os.makedirs(NQ_DIR, exist_ok=True)

def save_jsonl(df, filename, split_name):
    path = os.path.join(NQ_DIR, filename)

    with open(path, "w", encoding="utf-8") as f:
        for idx, row in df.iterrows():

            record = {
                "id": str(idx),
                "question": row["question"],
                "answers": [row["short_answer"]],
                "long_answer": row["long_answer"],
                "split": split_name
            }

            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print("Saved:", path)

save_jsonl(train_df, "train.jsonl", "train")
save_jsonl(val_df, "validation.jsonl", "validation")
save_jsonl(test_df, "test.jsonl", "test")

Saved: /kaggle/working/search_rl_project/nq_splits/train.jsonl
Saved: /kaggle/working/search_rl_project/nq_splits/validation.jsonl
Saved: /kaggle/working/search_rl_project/nq_splits/test.jsonl


In [18]:
from rank_bm25 import BM25Okapi
import re
import json
from tqdm.auto import tqdm

wiki_records = read_jsonl(WIKI_PASSAGES_PATH)

print("Total Wikipedia passages:", len(wiki_records))

wiki_texts = [
    record["text"]
    for record in wiki_records
]

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

tokenized_corpus = [
    tokenize(text)
    for text in tqdm(wiki_texts)
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index created successfully!")

Total Wikipedia passages: 469288


  0%|          | 0/469288 [00:00<?, ?it/s]

BM25 index created successfully!


In [19]:
def search_wikipedia(query, top_k=3):

    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:
        record = wiki_records[idx]

        results.append({
            "id": record["id"],
            "title": record["title"],
            "text": record["text"],
            "score": float(scores[idx])
        })

    return results

In [20]:
query = "Who played the mother in How I Met Your Mother?"

results = search_wikipedia(query, top_k=3)

for i, result in enumerate(results, 1):
    print(f"\nResult {i}")
    print("Title:", result["title"])
    print("Score:", result["score"])
    print("Text:", result["text"][:500])


Result 1
Title: Father's Day (disambiguation)
Score: 43.43574470407652
Text: tornado, also known as the Father's Day Tornado Les Compères, a 1983 French film remade as the 1997 film Father's Day Stepfather III, or Stepfather III: Father's Day in home video releases "Not a Father's Day", an episode of How I Met Your Mother Parents' Day, a holiday combining Father's Day and Mother's Day Mother's Day (disambiguation)

Result 2
Title: Like a Surgeon
Score: 40.05071711387205
Text: show How I Met Your Mother. In one episode, lead character Ted Mosby gave "Weird Al" the idea to the song after sending him a fan letter when he was 8 years old. The creation of "Like a Surgeon", partially from Madonna's suggestion, became part of several scenes in the 2010 satirical Funny or Die web short, "Weird: The Al Yankovic Story" in which Yankovic (played by Aaron Paul) ends up in a love affair with Madonna (played by Olivia Wilde). The short, including this love affair, was later expand

Result 3
Title: 

In [21]:
import os
import json
import time

PROJECT_DIR = "/kaggle/working/search_rl_project"

manifest = {
    "project": "Search-R1-inspired QA with RL",
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "base_model": "Qwen/Qwen2.5-0.5B-Instruct",

    "nq": {
        "source": "Natural-Questions-Base.csv",
        "source_records": 307373,
        "questions_with_short_answers": 106915,
        "train_records": len(train_df),
        "validation_records": len(val_df),
        "test_records": len(test_df),
        "random_state": 42,
        "split": "90/5/5 after filtering empty short answers"
    },

    "wikipedia": {
        "source": "wikimedia/wikipedia",
        "config": WIKI_CONFIG,
        "articles_processed": 50000,
        "passages": len(wiki_records),
        "passage_words": PASSAGE_WORDS,
        "overlap_words": OVERLAP_WORDS
    },

    "retrieval": {
        "method": "BM25",
        "top_k": 3,
        "tokenization": "lowercase alphanumeric regex"
    },

    "training": {
        "started": False,
        "previous_checkpoints_used": False
    }
}

manifest_path = os.path.join(PROJECT_DIR, "project_manifest.json")

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("Manifest saved:", manifest_path)

Manifest saved: /kaggle/working/search_rl_project/project_manifest.json


In [22]:
required_files = [
    WIKI_PASSAGES_PATH,
    os.path.join(NQ_DIR, "train.jsonl"),
    os.path.join(NQ_DIR, "validation.jsonl"),
    os.path.join(NQ_DIR, "test.jsonl"),
    manifest_path
]

print("PROJECT FILE CHECK")
print("=" * 60)

for path in required_files:
    exists = os.path.exists(path)

    print(
        "OK" if exists else "MISSING",
        path
    )

    if exists:
        print(
            "Size:",
            round(os.path.getsize(path) / (1024**2), 2),
            "MB"
        )

PROJECT FILE CHECK
OK /kaggle/working/search_rl_project/wiki/wikipedia_passages.jsonl
Size: 354.56 MB
OK /kaggle/working/search_rl_project/nq_splits/train.jsonl
Size: 119.87 MB
OK /kaggle/working/search_rl_project/nq_splits/validation.jsonl
Size: 6.72 MB
OK /kaggle/working/search_rl_project/nq_splits/test.jsonl
Size: 6.56 MB
OK /kaggle/working/search_rl_project/project_manifest.json
Size: 0.0 MB


In [23]:
import zipfile
import os

BACKUP_PATH = "/kaggle/working/search_rl_pretraining_backup.zip"

files_to_backup = [
    WIKI_PASSAGES_PATH,
    os.path.join(NQ_DIR, "train.jsonl"),
    os.path.join(NQ_DIR, "validation.jsonl"),
    os.path.join(NQ_DIR, "test.jsonl"),
    manifest_path
]

with zipfile.ZipFile(
    BACKUP_PATH,
    "w",
    compression=zipfile.ZIP_STORED
) as zipf:

    for path in files_to_backup:
        if os.path.exists(path):
            arcname = os.path.relpath(path, PROJECT_DIR)
            zipf.write(path, arcname=arcname)

print("Backup created:", BACKUP_PATH)
print(
    "Backup size (MB):",
    round(os.path.getsize(BACKUP_PATH) / (1024**2), 2)
)

Backup created: /kaggle/working/search_rl_pretraining_backup.zip
Backup size (MB): 487.7
